# Lecke 11 - Model Context Protocol (MCP)

A **Model Context Protocol (MCP)** egy nyílt szabvány, amely lehetővé teszi az ügynökök számára, hogy futásidőben dinamikusan felfedezzék és használják az eszközöket, erőforrásokat és adatforrásokat. Ahelyett, hogy az eszközöket keménykódolnánk az ügynökbe, az MCP lehetővé teszi az ügynökök számára, hogy külső szerverekhez csatlakozzanak, amelyek igény szerint tárják fel a képességeket.

Ebben a leckében megtanulod:
- Mi az MCP és miért fontos az ügynökrendszerek számára
- Hogyan működik az MCP kliens-szerver architektúrája
- Hogyan építsünk olyan ügynököket, amelyek MCP-stílusú eszközfelfedezést használnak


## Beállítás

**Előfeltételek:**
- Azure AI Foundry projekt telepített modellel
- Futtassa az `az login` parancsot a hitelesítéshez


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mi az a Model Context Protocol (MCP)?

A MCP egy szabványos módot határoz meg a mesterséges intelligencia (AI) ügynökök számára, hogy felfedezzék és kölcsönhatásba lépjenek külső eszközökkel és adatforrásokkal:

- **MCP Server**: Eszközöket, erőforrásokat és promptokat tesz elérhetővé egy szabványos protokollon keresztül
- **MCP Client**: Az ügynök futtatókörnyezete, amely csatlakozik a szerverekhez és felfedezi a rendelkezésre álló képességeket
- **Dynamic Discovery**: Az ügynököknek nincs szükségük keménykódolt eszközökre — futásidőben fedezik fel, mi érhető el

Ez erőteljes megoldás kiterjeszthető ügynök-rendszerek építéséhez, ahol új képességek adhatók hozzá az ügynök kódjának módosítása nélkül.


## Hogyan működik az MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Az ügynök (MCP client) kapcsolódik egy MCP szerverhez
2. A szerver egy listával válaszol az elérhető eszközökről és azok sémáiról
3. Az ügynök ezután meghívhat bármely felfedezett eszközt a következtetése során
4. Az eredmények ugyanazon a protokollon keresztül érkeznek vissza


## MCP-eszköz felfedezésének szimulálása

Mivel egy valódi MCP szerver futó szerverfolyamatot igényel, a mintát `@tool` függvények használatával mutatjuk be, amelyek szimulálják azt, amit egy MCP-hez csatlakoztatott szállásszolgáltatás biztosítana.

Éles környezetben ezeket az eszközöket egy MCP szerverről dinamikusan fedeznék fel, ahelyett, hogy helyben definiálnánk.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Ügynök létrehozása MCP-stílusú eszközökkel


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP éles környezetben

Éles környezetben az MCP erőteljes mintákat tesz lehetővé:

- **Dinamikus eszközfelismerés**: Az ügynökök csatlakoznak az MCP szerverekhez, és futásidőben fedezik fel az eszközöket
- **Laza csatolású architektúra**: Az eszközszolgáltatók az ügynöktől függetlenül frissíthetnek
- **Szervezetek közötti megosztás**: A csapatok képességeket tehetnek elérhetővé MCP szervereken keresztül, amelyeket bármely ügynök használhat
- **Microsoft Agent Framework támogatás**: A MAF beépített MCP kliens támogatással rendelkezik az `mcp` integráción keresztül

Ha egy valós MCP szervert szeretnél használni a MAF-pal, csatlakozhatsz a `hosted_mcp_tool()`-on keresztül vagy az MCP kliens integráción keresztül.

**További információk:**
- [MCP specifikáció](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP támogatás](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Összefoglalás

Ebben a leckében megtanultad:
- **MCP** egy nyílt szabvány az ügynökök és eszközszolgáltatók közötti dinamikus eszközfelismerésre
- A **kliens-szerver architektúra** lehetővé teszi, hogy az ügynökök futásidőben felfedezzék a képességeiket
- MCP lehetővé teszi **kiterjeszthető, lazán csatolt ügynökrendszereket**, ahol az eszközök kódmódosítás nélkül adhatók hozzá
- Microsoft Agent Framework biztosít **beépített MCP-támogatást** éles használatra


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Felelősségkizárás:
Ezt a dokumentumot az AI-alapú fordító szolgáltatás, a Co-op Translator (https://github.com/Azure/co-op-translator) használatával fordítottuk. Bár törekszünk a pontosságra, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Kritikus információk esetén professzionális, emberi fordítást javaslunk. Nem vállalunk felelősséget a fordítás használatából eredő félreértésekért vagy téves értelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
